# Deep Learning — Module 2: Training & Optimization · Part 2
## Optimizers & Learning Rate Scheduling

> Every concept in three layers: **Intuition → Math → Code**

---

## Table of Contents

| Section | Topic |
|---------|-------|
| **1** | Gradient Descent Variants (Batch, Mini-batch, SGD) |
| **2** | Momentum — Escaping Ravines |
| **3** | AdaGrad — Per-parameter Learning Rates |
| **4** | RMSProp — Fixing AdaGrad's Fading Memory |
| **5** | Adam — The Industry Default |
| **6** | AdamW — Weight Decay Done Right |
| **7** | Optimizer Trajectory Visualization |
| **8** | LR Scheduling (Step, Cosine, Warmup, LR Finder) |
| **9** | PyTorch Optimizer & Scheduler Demo |
| **10** | Master Interview Q&A Cheatsheet |


In [ ]:
# ── All imports — run this first ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn as nn
import torch.optim as optim

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "axes.prop_cycle": plt.cycler(color=[
        "#4C72B0","#DD8452","#55A868","#C44E52","#8172B3","#937860","#DA8BC3","#8C8C8C"
    ])
})
print("Imports ready ✓")


## 1. Gradient Descent Variants

### Intuition: Hiking Down a Mountain
You're hiking down a misty mountain (the loss landscape) trying to reach the valley (minimum loss).

Three strategies:
- **Batch GD** — look at the *entire map* before each step → most accurate direction, but very slow
- **Stochastic GD (SGD)** — take a step based on *one random observation* → fast but noisy
- **Mini-batch GD** — look at *a small sample* of the map → best of both worlds ✓

---

### Math

Let $W$ = current weights, $\eta$ = learning rate, $n$ = dataset size, $B$ = batch size.

#### Batch Gradient Descent
$$W \leftarrow W - \eta \cdot \frac{1}{n}\sum_{i=1}^{n}\nabla_W \mathcal{L}(x_i, y_i)$$
- Uses **all** $n$ examples per update
- Exact gradient → smooth convergence
- Impractical: won't fit ImageNet in RAM

#### Stochastic Gradient Descent (SGD)
$$W \leftarrow W - \eta \cdot \nabla_W \mathcal{L}(x_i, y_i) \quad \text{(one random sample)}$$
- **One** example per update → very noisy gradient estimate
- Fast but oscillates wildly
- The noise can help escape local minima

#### Mini-batch GD (what everyone calls "SGD" in practice)
$$W \leftarrow W - \eta \cdot \frac{1}{B}\sum_{i \in \text{batch}}\nabla_W \mathcal{L}(x_i, y_i)$$
- $B$ examples per update (typical: 32, 64, 128, 256)
- Good gradient estimate + GPU parallelism + memory feasible

| Variant | Gradient Quality | Speed | Memory | Noise |
|---------|-----------------|-------|--------|-------|
| Batch GD | ✅ Exact | ❌ Slow | ❌ High | ✅ None |
| SGD | ❌ Very noisy | ✅ Fast | ✅ Low | ❌ High |
| Mini-batch | ✅ Good approx | ✅ Fast | ✅ Medium | ✅ Moderate |

#### The Learning Rate Problem
Vanilla SGD has one global learning rate $\eta$ for all parameters. Problems:
1. Sparse features need larger updates; dense ones need smaller
2. Loss landscape may be steep in one direction and flat in another
3. Fixed $\eta$: either too fast (diverges) or too slow (stalls)

→ Everything from Section 2 onward fixes one or more of these problems.

#### Interview Questions: SGD
> **Q: Why do we use mini-batches instead of true SGD (batch=1)?**
> A: GPU parallelism — processing 32 samples simultaneously is nearly as fast as 1 sample on GPU. Also: better gradient estimates, less oscillation, faster wall-clock convergence.

> **Q: How does batch size affect training dynamics?**
> A: Small batches: noisy gradients, implicit regularisation, may find flatter minima. Large batches: accurate gradients, faster but may converge to sharper (worse-generalising) minima. "Linear scaling rule": if you double batch size, double learning rate.

> **Q: Can vanilla SGD get stuck in local minima?**
> A: In theory yes, but in practice deep networks have very few bad local minima — most critical points are saddle points. The real issue is saddle points and plateaus, not local minima.


In [ ]:
# Figure 1: SGD — batch size effect on gradient noise & convergence
np.random.seed(42)

# Synthetic 1D regression
N = 500
X_data = np.random.randn(N)
Y_data = 2.5 * X_data + np.random.randn(N) * 0.8

def mse_grad(w, x_batch, y_batch):
    pred = w * x_batch
    return 2 * np.mean((pred - y_batch) * x_batch)

def train_gd(batch_size, lr=0.05, epochs=80):
    w = 0.0
    loss_hist = []
    for _ in range(epochs):
        idx = np.random.permutation(N)
        for start in range(0, N, batch_size):
            batch = idx[start:start+batch_size]
            g = mse_grad(w, X_data[batch], Y_data[batch])
            w -= lr * g
        pred = w * X_data
        loss_hist.append(np.mean((pred - Y_data)**2))
    return loss_hist

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

configs = [(1,'SGD (B=1)','#C44E52'), (32,'Mini-batch (B=32)','#55A868'),
           (N,'Batch GD (B=N)','#4C72B0')]
for bs, label, col in configs:
    h = train_gd(bs)
    axes[0].plot(h, label=label, color=col, lw=2, alpha=0.85)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Effect of Batch Size on Loss Curve'); axes[0].legend()

# Weight update path for B=1 vs B=32
for bs, label, col in [(1,'SGD (B=1)','#C44E52'), (32,'Mini-batch (B=32)','#55A868')]:
    np.random.seed(0)
    w, w_path = 0.0, [0.0]
    for _ in range(40):
        idx = np.random.permutation(N)
        batch = idx[:bs]
        g = mse_grad(w, X_data[batch], Y_data[batch])
        w -= 0.05 * g
        w_path.append(w)
    axes[1].plot(w_path, label=label, color=col, lw=2, alpha=0.85)

axes[1].axhline(2.5, color='grey', ls='--', lw=1.5, label='True w=2.5')
axes[1].set_xlabel('Update step'); axes[1].set_ylabel('Weight w')
axes[1].set_title('Weight Trajectory: SGD Noise vs Mini-batch'); axes[1].legend()

plt.suptitle('Figure 1: SGD Variants — Batch Size Effect', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 2. Momentum — Escaping Ravines

### Intuition: A Ball Rolling Down a Hill
Vanilla SGD is like taking tiny individual steps, re-reading the map each time.

**Momentum** is like a ball rolling downhill — it **accumulates velocity** in directions it keeps moving, and dampens oscillations in directions it keeps flipping.

Imagine a ball in a narrow ravine (loss landscape where one axis is very steep, the other very flat):
- Vanilla SGD: zig-zags wildly across the ravine, barely moving forward
- Momentum SGD: accumulates speed in the forward direction, dampens the zig-zag

---

### Math

Introduce a **velocity vector** $v$ (same shape as $W$):

$$v_t = \beta v_{t-1} + (1-\beta)\nabla_W\mathcal{L}_t \quad\text{(some formulations omit }(1-\beta)\text{)}$$
$$W_t = W_{t-1} - \eta\, v_t$$

Common alternative (PyTorch default):
$$v_t = \beta v_{t-1} + \nabla_W\mathcal{L}_t$$
$$W_t = W_{t-1} - \eta\, v_t$$

- $\beta$ = momentum coefficient (typical: 0.9)
- $v_0 = 0$ (no initial velocity)
- When $\beta = 0$: reduces to vanilla SGD

#### What momentum does geometrically
- **Consistent gradient direction** → velocity builds up → faster progress (like terminal velocity)
- **Oscillating gradient direction** → velocity cancels out → dampened zig-zag

#### Nesterov Momentum (the smarter variant)
Compute gradient *at the anticipated next position*:

$$v_t = \beta v_{t-1} + \nabla_W\mathcal{L}(W - \beta v_{t-1})$$
$$W_t = W_{t-1} - \eta\, v_t$$

PyTorch: `SGD(lr=0.01, momentum=0.9, nesterov=True)`

Nesterov "looks ahead" — corrects course before the mistake, not after.

#### Interview Questions: Momentum
> **Q: What does the β hyperparameter control?**
> A: How much of the previous velocity is retained. β=0.9 means 90% of old velocity is kept. High β → smooth trajectory, more momentum. Low β → closer to vanilla SGD.

> **Q: Why does momentum help with saddle points?**
> A: At a saddle point, gradient is near 0 in some directions. Accumulated velocity from previous steps carries the optimizer through the flat region.

> **Q: What is Nesterov momentum and why is it better?**
> A: Standard momentum computes the gradient at the current position, then jumps. Nesterov jumps first (using velocity) then corrects — this "look-ahead" reduces overshoot and converges faster in theory.


In [ ]:
# Figure 2: Momentum vs SGD — trajectory through a ravine loss surface
np.random.seed(7)

# Ravine loss: steep in y, flat in x — classic momentum test
def ravine_loss(w1, w2): return w1**2 * 0.1 + w2**2 * 10
def ravine_grad(w1, w2): return np.array([0.2*w1, 20*w2])

def run_sgd(lr=0.1, steps=50):
    w = np.array([2.5, 0.8])
    path = [w.copy()]
    for _ in range(steps):
        g = ravine_grad(*w)
        w = w - lr * g
        path.append(w.copy())
    return np.array(path)

def run_momentum(lr=0.1, beta=0.9, steps=50):
    w = np.array([2.5, 0.8])
    v = np.zeros(2)
    path = [w.copy()]
    for _ in range(steps):
        g = ravine_grad(*w)
        v = beta * v + g
        w = w - lr * v
        path.append(w.copy())
    return np.array(path)

path_sgd = run_sgd(lr=0.09)
path_mom = run_momentum(lr=0.09, beta=0.9)

# Loss surface
w1 = np.linspace(-3, 3, 150)
w2 = np.linspace(-1.2, 1.2, 150)
W1, W2 = np.meshgrid(w1, w2)
L = ravine_loss(W1, W2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, path, title, col in [
    (axes[0], path_sgd, 'Vanilla SGD — zig-zag ravine', '#C44E52'),
    (axes[1], path_mom, 'Momentum (β=0.9) — smooth trajectory', '#4C72B0')
]:
    ax.contourf(W1, W2, L, levels=25, cmap='RdYlGn_r', alpha=0.75)
    ax.contour(W1, W2, L, levels=25, colors='white', alpha=0.2, linewidths=0.4)
    pts = path[:30]
    ax.plot(pts[:,0], pts[:,1], 'o-', ms=4, lw=2, color=col, label=title, zorder=5)
    ax.plot(pts[0,0], pts[0,1], 'b^', ms=12, label='Start')
    ax.plot(0, 0, 'r*', ms=14, label='Minimum (0,0)')
    ax.set_xlabel('w₁ (flat direction)'); ax.set_ylabel('w₂ (steep direction)')
    ax.set_title(title); ax.legend(fontsize=9)

plt.suptitle('Figure 2: Momentum Smooths the Zig-Zag', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Show step count to same loss threshold
threshold = 0.01
def steps_to_converge(path):
    for i,(w1,w2) in enumerate(path):
        if ravine_loss(w1,w2) < threshold: return i
    return len(path)

print(f"Steps for SGD to reach loss < {threshold}: {steps_to_converge(path_sgd)}")
print(f"Steps for Momentum to reach loss < {threshold}: {steps_to_converge(path_mom)}")


## 3. AdaGrad — Per-parameter Learning Rates

### Intuition: Learning at Different Speeds for Different Features
Some features in your dataset appear frequently (common words in NLP, popular products in recommendations).
Others appear rarely. A single global learning rate treats them identically:
- Frequent features: get updated too aggressively → overshoot
- Rare features: get updated too conservatively → they barely learn

**AdaGrad** solves this by giving **each parameter its own learning rate**, scaled inversely to how often it has been updated.

---

### Math

Maintain a running sum of squared gradients $G$ (same shape as $W$):

$$G_t = G_{t-1} + (\nabla_W\mathcal{L}_t)^2 \quad\text{(elementwise square)}$$

$$W_t = W_{t-1} - \frac{\eta}{\sqrt{G_t + \epsilon}} \cdot \nabla_W\mathcal{L}_t$$

- $\epsilon \approx 10^{-8}$: small constant to avoid division by zero
- Parameters with large accumulated gradients → denominator grows → **smaller effective lr**
- Parameters with small accumulated gradients → denominator stays small → **larger effective lr**

#### The Fading Memory Problem
$G_t$ only grows — it **never shrinks**.

For a parameter updated frequently, $G_t → \infty$, so the effective learning rate → 0.

After training long enough, **all learning stops.** This is AdaGrad's fatal flaw for deep learning.

✅ Great for sparse data (NLP, recommendations)
❌ Bad for long training runs — learning rate decays to 0

#### Interview Questions: AdaGrad
> **Q: How does AdaGrad adapt learning rates per parameter?**
> A: It divides the learning rate by $\sqrt{\text{sum of squared past gradients}}$. Parameters with historically large gradients get smaller effective lr; sparse params with small gradients get larger effective lr.

> **Q: What is AdaGrad's main weakness?**
> A: The accumulated squared gradient $G_t$ only ever grows (monotonically). For frequently updated parameters, the effective learning rate decays to 0, causing training to stall. RMSProp fixes this with exponential moving average.


In [ ]:
# Figure 3: AdaGrad — per-parameter lr adaptation + fading lr problem
np.random.seed(0)

def run_adagrad(lr=0.5, steps=200, eps=1e-8):
    w = np.array([2.5, 0.8])
    G = np.zeros(2)
    path, eff_lr_hist = [w.copy()], []
    for _ in range(steps):
        g = ravine_grad(*w)
        G += g**2
        eff_lr = lr / (np.sqrt(G) + eps)
        w = w - eff_lr * g
        path.append(w.copy())
        eff_lr_hist.append(eff_lr.copy())
    return np.array(path), np.array(eff_lr_hist)

path_ada, eff_lrs = run_adagrad(lr=0.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: Trajectory ──
axes[0].contourf(W1, W2, L, levels=25, cmap='RdYlGn_r', alpha=0.75)
axes[0].contour(W1, W2, L, levels=25, colors='white', alpha=0.2, linewidths=0.4)
axes[0].plot(path_ada[:60,0], path_ada[:60,1], 'o-', ms=3, lw=2, color='#8172B3')
axes[0].plot(path_ada[0,0], path_ada[0,1], 'b^', ms=12)
axes[0].plot(0, 0, 'r*', ms=14)
axes[0].set_title('AdaGrad Trajectory'); axes[0].set_xlabel('w₁'); axes[0].set_ylabel('w₂')

# ── Plot 2: Effective learning rate decay over time ──
axes[1].plot(eff_lrs[:,0], label='eff_lr for w₁', color='#4C72B0', lw=2)
axes[1].plot(eff_lrs[:,1], label='eff_lr for w₂', color='#DD8452', lw=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Effective learning rate')
axes[1].set_title('AdaGrad: Effective LR Decays to 0 (fading memory)')
axes[1].legend()

# ── Plot 3: Compare per-step update magnitude vs SGD ──
def run_sgd_steps(lr=0.09, steps=100):
    w = np.array([2.5, 0.8])
    updates = []
    for _ in range(steps):
        g = ravine_grad(*w)
        update = lr * g
        w -= update
        updates.append(np.linalg.norm(update))
    return updates

def run_adagrad_steps(lr=0.5, steps=100, eps=1e-8):
    w = np.array([2.5, 0.8])
    G = np.zeros(2)
    updates = []
    for _ in range(steps):
        g = ravine_grad(*w)
        G += g**2
        update = (lr / (np.sqrt(G) + eps)) * g
        w -= update
        updates.append(np.linalg.norm(update))
    return updates

axes[2].plot(run_sgd_steps(),      label='SGD update size',     color='#C44E52', lw=2)
axes[2].plot(run_adagrad_steps(),  label='AdaGrad update size', color='#8172B3', lw=2)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('||update|| norm')
axes[2].set_title('Update Magnitude: SGD (constant) vs AdaGrad (fading)')
axes[2].legend()

plt.suptitle('Figure 3: AdaGrad — Adaptive LR with Fading Memory', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 4. RMSProp — Fixing AdaGrad's Fading Memory

### Intuition: Forget the Distant Past
AdaGrad's problem: it remembers every gradient it ever saw, equally.
Old gradients from early training (when the model was wrong in a different way) keep shrinking the learning rate.

**RMSProp** says: *"Only care about recent gradients."*

It replaces the running sum with an **exponential moving average** (EMA) — recent gradients count more, old ones fade away.

---

### Math

$$E[g^2]_t = \rho\, E[g^2]_{t-1} + (1-\rho)(\nabla_W\mathcal{L}_t)^2$$

$$W_t = W_{t-1} - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \cdot \nabla_W\mathcal{L}_t$$

- $\rho$ = decay rate (typical: 0.9 or 0.99)
- $E[g^2]_t$ = exponential moving average of squared gradients
- **The key difference from AdaGrad:** $E[g^2]_t$ can decrease if recent gradients are small

#### Why EMA fixes the fading problem
In AdaGrad: $G_t = G_0 + g_1^2 + g_2^2 + ... + g_t^2$ → always grows
In RMSProp: $E[g^2]_t \approx (1-\rho)\sum_{k=0}^{t}\rho^{t-k}g_k^2$ → exponential weights, recent grads dominate

If gradients become small, $E[g^2]_t$ shrinks → effective learning rate can grow again → **training doesn't stall**.

#### When to use RMSProp
- Default choice for **RNNs** (recommended by Hinton in his original lecture)
- Good when gradients are non-stationary (change character over training)

#### Interview Questions: RMSProp
> **Q: How does RMSProp differ from AdaGrad?**
> A: AdaGrad accumulates all squared gradients equally. RMSProp uses an exponential moving average — recent gradients get exponentially higher weight, old gradients fade. This prevents the effective learning rate from decaying to zero.

> **Q: What does the ρ (rho/decay) parameter control?**
> A: How quickly old gradients are forgotten. ρ=0.9 means: current EMA = 90% old + 10% new. Higher ρ → longer memory. Lower ρ → adapts faster to recent gradient changes.


In [ ]:
# Figure 4: RMSProp — effective LR stays alive vs AdaGrad
np.random.seed(0)

def run_rmsprop(lr=0.1, rho=0.9, steps=200, eps=1e-8):
    w = np.array([2.5, 0.8])
    E_g2 = np.zeros(2)
    path, eff_lr_hist = [w.copy()], []
    for _ in range(steps):
        g = ravine_grad(*w)
        E_g2 = rho * E_g2 + (1 - rho) * g**2
        eff_lr = lr / (np.sqrt(E_g2) + eps)
        w = w - eff_lr * g
        path.append(w.copy())
        eff_lr_hist.append(eff_lr.copy())
    return np.array(path), np.array(eff_lr_hist)

path_rms, eff_lrs_rms = run_rmsprop(lr=0.1)
_, eff_lrs_ada = run_adagrad(lr=0.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: Trajectory comparison ──
axes[0].contourf(W1, W2, L, levels=25, cmap='RdYlGn_r', alpha=0.75)
axes[0].contour(W1, W2, L, levels=25, colors='white', alpha=0.2, linewidths=0.4)
for path, col, lbl in [(path_ada,'#8172B3','AdaGrad'),(path_rms,'#DD8452','RMSProp')]:
    axes[0].plot(path[:50,0], path[:50,1], 'o-', ms=3, lw=2, color=col, label=lbl, alpha=0.85)
axes[0].plot(path_ada[0,0], path_ada[0,1], 'b^', ms=12)
axes[0].plot(0, 0, 'r*', ms=14)
axes[0].set_title('Trajectories: AdaGrad vs RMSProp')
axes[0].set_xlabel('w₁'); axes[0].set_ylabel('w₂'); axes[0].legend()

# ── Plot 2: Effective LR comparison ──
axes[1].plot(eff_lrs_ada[:,0], label='AdaGrad eff_lr(w₁)', color='#8172B3', lw=2, ls='--')
axes[1].plot(eff_lrs_rms[:,0], label='RMSProp eff_lr(w₁)', color='#DD8452', lw=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Effective learning rate')
axes[1].set_title('AdaGrad LR → 0  vs  RMSProp LR stays alive')
axes[1].legend()

# ── Plot 3: EMA of g² vs cumsum of g² ──
g_seq = np.abs(np.sin(np.linspace(0, 6*np.pi, 200))) + 0.1
adagrad_denom = np.cumsum(g_seq**2)
rms_denom = np.zeros(200)
v = 0
for i,g in enumerate(g_seq):
    v = 0.9*v + 0.1*g**2
    rms_denom[i] = v
axes[2].plot(np.sqrt(adagrad_denom), label='AdaGrad √G (grows forever)', color='#8172B3', lw=2)
axes[2].plot(np.sqrt(rms_denom), label='RMSProp √E[g²] (bounded)', color='#DD8452', lw=2)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Denominator value')
axes[2].set_title('Why RMSProp Denominator Stays Bounded')
axes[2].legend()

plt.suptitle('Figure 4: RMSProp vs AdaGrad — EMA Fixes Fading LR', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 5. Adam — The Industry Default

### Intuition: Momentum + RMSProp, Bias-Corrected
Adam = **Ada**ptive **M**oment estimation.

It combines two ideas:
- **Momentum** (1st moment): tracks direction using EMA of gradients
- **RMSProp** (2nd moment): adapts learning rate using EMA of squared gradients

Plus a crucial fix: **bias correction** for the cold start.

Think of it as: *"I know where I've been going (momentum), and I know how bumpy the terrain has been (RMSProp). I'll take a confident step in the smooth direction and a cautious step in the bumpy direction."*

---

### Math

**Maintain two moment estimates:**

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad\text{(1st moment: EMA of gradients)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2 \quad\text{(2nd moment: EMA of squared gradients)}$$

**Bias correction** (both start at 0, so early estimates are biased toward 0):
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**Update rule:**
$$W_t = W_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

**Default hyperparameters** (almost always kept as-is):
- $\eta = 0.001$ (learning rate)
- $\beta_1 = 0.9$ (momentum decay)
- $\beta_2 = 0.999$ (RMSProp decay)
- $\epsilon = 10^{-8}$

#### Why Bias Correction Matters
At $t=1$: $m_1 = 0.9 \cdot 0 + 0.1 \cdot g_1 = 0.1 g_1$

Without correction: $m_1 = 0.1 g_1$ (biased 10× too small!)
With correction:   $\hat{m}_1 = m_1 / (1 - 0.9^1) = m_1 / 0.1 = g_1$ ✓

As $t → \infty$ the correction factor $1-\beta^t → 1$, so it has no effect after warmup.

#### Intuition for the effective step size
$$\text{effective step} = \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

If $|g|$ is consistent (always positive): $\hat{m}_t \approx g$, $\hat{v}_t \approx g^2$, so step $\approx \eta$ — bounded step size.
If $g$ oscillates (sign flips): $\hat{m}_t → 0$, but $\hat{v}_t > 0$ → step → 0 — correct: don't move in noisy directions.

#### Interview Questions: Adam
> **Q: What are the two moments Adam tracks and what do they mean?**
> A: 1st moment ($m_t$): exponential moving average of gradients — captures direction/momentum. 2nd moment ($v_t$): EMA of squared gradients — captures gradient magnitude/variance. Both are used together to form an adaptive, momentum-based update.

> **Q: Why is bias correction necessary in Adam?**
> A: Both moments are initialised to 0. In early steps, EMA is strongly biased toward 0. Dividing by $(1-\beta^t)$ corrects this — scaling up the small initial estimates to their true value.

> **Q: Adam vs SGD with momentum — which is better?**
> A: It depends. Adam converges faster early on and requires less LR tuning. SGD+momentum often achieves better final test accuracy with careful tuning. In practice: use Adam for fast experimentation; switch to SGD+momentum for production CV models if you have compute.

> **Q: What is Adam's effective learning rate bounded by?**
> A: Approximately $\eta$. The ratio $|\hat{m}|/\sqrt{\hat{v}}$ is bounded by $1/\sqrt{1-\beta_2}$ — with $\beta_2=0.999$, this is ~31.6, so effective step is within $\approx 31.6 \eta$. In practice it's usually much smaller.


In [ ]:
# Figure 5: Adam — moment tracking, bias correction, vs SGD/Momentum/RMSProp
np.random.seed(7)

def run_adam(lr=0.1, b1=0.9, b2=0.999, eps=1e-8, steps=80):
    w = np.array([2.5, 0.8])
    m, v = np.zeros(2), np.zeros(2)
    path = [w.copy()]
    for t in range(1, steps+1):
        g = ravine_grad(*w)
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(w.copy())
    return np.array(path)

path_adam = run_adam(lr=0.3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: All trajectories together ──
axes[0].contourf(W1, W2, L, levels=25, cmap='RdYlGn_r', alpha=0.75)
axes[0].contour(W1, W2, L, levels=25, colors='white', alpha=0.2, linewidths=0.4)
for path, col, lbl in [
    (path_sgd,  '#C44E52', 'SGD'),
    (path_mom,  '#4C72B0', 'Momentum'),
    (path_rms,  '#DD8452', 'RMSProp'),
    (path_adam, '#55A868', 'Adam'),
]:
    pts = path[:40]
    axes[0].plot(pts[:,0], pts[:,1], 'o-', ms=3, lw=2, color=col, label=lbl, alpha=0.85)
axes[0].plot(path_sgd[0,0], path_sgd[0,1], 'b^', ms=10)
axes[0].plot(0, 0, 'r*', ms=14)
axes[0].legend(fontsize=8); axes[0].set_title('All Optimizers: Trajectory Comparison')
axes[0].set_xlabel('w₁'); axes[0].set_ylabel('w₂')

# ── Plot 2: Bias correction demonstration ──
b1 = 0.9
g_seq = np.ones(50)   # constant gradient = 1
m_raw, m_corrected = [], []
m = 0.0
for t in range(1, 51):
    m = b1*m + (1-b1)*g_seq[t-1]
    m_raw.append(m)
    m_corrected.append(m / (1 - b1**t))
axes[1].plot(m_raw,       label='m_t (raw, biased)', color='#C44E52', lw=2)
axes[1].plot(m_corrected, label='m̂_t (bias-corrected)', color='#55A868', lw=2)
axes[1].axhline(1.0, color='grey', ls='--', lw=1.5, label='True gradient = 1')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Moment estimate')
axes[1].set_title('Bias Correction: Cold Start Fix'); axes[1].legend()

# ── Plot 3: Steps to convergence comparison ──
threshold = 0.005
def conv_steps(path):
    for i,(w1,w2) in enumerate(path):
        if ravine_loss(w1,w2) < threshold: return i
    return len(path)

opt_names = ['SGD','Momentum','AdaGrad','RMSProp','Adam']
paths_all  = [path_sgd, path_mom, path_ada, path_rms, path_adam]
steps_all  = [conv_steps(p) for p in paths_all]
colors_all = ['#C44E52','#4C72B0','#8172B3','#DD8452','#55A868']
bars = axes[2].bar(opt_names, steps_all, color=colors_all, alpha=0.88)
axes[2].set_ylabel('Steps to convergence'); axes[2].set_title(f'Steps to Loss < {threshold}')
for bar, v in zip(bars, steps_all):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 str(v) if v < 200 else '>200', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Figure 5: Adam — Combining Momentum + RMSProp', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 6. AdamW — Weight Decay Done Right

### The Problem with L2 Regularisation + Adam

In vanilla Adam, applying L2 regularisation (adding $\lambda W$ to the gradient) **doesn't work as intended**.

Here's why:

In **SGD**, L2 regularisation and weight decay are equivalent:
$$W_t = W_{t-1} - \eta(g_t + \lambda W_{t-1}) = W_{t-1}(1-\eta\lambda) - \eta g_t$$

Adding $\lambda W$ to the gradient is the same as decaying the weight by $(1-\eta\lambda)$.

In **Adam**, the adaptive denominator $\sqrt{\hat{v}_t}$ also adapts the regularisation term:
$$W_t = W_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t}}(g_t + \lambda W_{t-1})$$

The weight decay term $\lambda W$ is **scaled by the adaptive step** — parameters with small gradients (but may be large weights) get much less regularisation than intended.

---

### AdamW Fix: Decouple Weight Decay

Apply weight decay **directly** to the weights, bypassing the adaptive denominator:

$$W_t = W_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t - \eta\lambda W_{t-1}$$

The $-\eta\lambda W_{t-1}$ term is applied uniformly to all parameters, regardless of gradient history.

Result: proper, consistent regularisation across all parameters.

#### In Practice
```python
# Adam with L2 = broken weight decay
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# AdamW = correct weight decay (use this!)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
```

AdamW is the **default for transformers** — BERT, GPT, LLaMA all train with AdamW.

#### Interview Questions: AdamW
> **Q: What is the difference between Adam and AdamW?**
> A: In Adam, L2 regularisation is applied as part of the gradient and hence gets scaled by the adaptive denominator. AdamW decouples weight decay — it's applied directly to weights at rate $\eta\lambda$, uniformly regardless of gradient history. This makes regularisation consistent across all parameters.

> **Q: Why is weight decay important for transformers?**
> A: Transformers have many parameters and tend to overfit. Consistent weight decay (via AdamW) prevents weights from growing unboundedly and improves generalisation. The original BERT paper specifically used AdamW over Adam.

> **Q: What is a good weight decay value to start with?**
> A: Typically 0.01 to 0.1 for transformers, 1e-4 to 1e-2 for CNNs. It's one of the most impactful hyperparameters after learning rate.


In [ ]:
# AdamW vs Adam — weight decay effect demonstration
import torch
import torch.nn as nn

torch.manual_seed(42)

# Simple model: 2-layer MLP on synthetic task
def make_model(): return nn.Sequential(nn.Linear(10,20), nn.ReLU(), nn.Linear(20,1))
def make_data(N=100):
    X = torch.randn(N, 10)
    y = torch.randn(N, 1)
    return X, y

X, y = make_data()

def train_optimizer(opt_class, opt_kwargs, steps=200):
    model = make_model()
    optimizer = opt_class(model.parameters(), **opt_kwargs)
    losses, weight_norms = [], []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = nn.MSELoss()(model(X), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        wn = sum(p.norm().item() for p in model.parameters())
        weight_norms.append(wn)
    return losses, weight_norms

loss_adam,  wn_adam  = train_optimizer(torch.optim.Adam,  {'lr':1e-2, 'weight_decay':0.1})
loss_adamw, wn_adamw = train_optimizer(torch.optim.AdamW, {'lr':1e-2, 'weight_decay':0.1})
loss_noreg, wn_noreg = train_optimizer(torch.optim.Adam,  {'lr':1e-2, 'weight_decay':0.0})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(loss_noreg, label='Adam (no weight decay)', color='#C44E52', lw=2)
axes[0].plot(loss_adam,  label='Adam (wd=0.1)',           color='#4C72B0', lw=2)
axes[0].plot(loss_adamw, label='AdamW (wd=0.1)',          color='#55A868', lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss: Adam vs AdamW'); axes[0].legend()

axes[1].plot(wn_noreg, label='Adam (no wd) — weights grow', color='#C44E52', lw=2)
axes[1].plot(wn_adam,  label='Adam (wd=0.1)',                color='#4C72B0', lw=2)
axes[1].plot(wn_adamw, label='AdamW (wd=0.1) — strongest regularisation', color='#55A868', lw=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Total weight norm')
axes[1].set_title('Weight Norm: AdamW Has Strongest Proper Decay'); axes[1].legend()

plt.suptitle('Figure 6: Adam vs AdamW — Weight Decay Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Show PyTorch API comparison
print("PyTorch API:")
print("  torch.optim.Adam(model.parameters(),  lr=1e-3, weight_decay=1e-4)   # L2 in gradient (less correct)")
print("  torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)   # True weight decay (preferred)")
print()
print("For transformers (BERT, GPT, ViT): AdamW is the standard ✓")


## 7. Optimizer Summary & Comparison

| Optimizer | Adapts LR? | Uses Momentum? | Memory Issue? | Best For |
|-----------|-----------|----------------|---------------|----------|
| **SGD** | ❌ | ❌ | ❌ | Baseline |
| **SGD+Momentum** | ❌ | ✅ | ❌ | CNNs with tuned LR |
| **AdaGrad** | ✅ | ❌ | ❌ Fades to 0 | Sparse NLP data |
| **RMSProp** | ✅ | ❌ | ✅ Fixed | RNNs, non-stationary |
| **Adam** | ✅ | ✅ | ✅ Fixed | Most deep learning |
| **AdamW** | ✅ | ✅ | ✅ Fixed | Transformers (preferred) |

### The 1st vs 2nd Moment Cheatsheet

| Moment | Symbol | Tracks | Used For |
|--------|--------|--------|---------|
| 1st moment (mean) | $m_t$ | Direction of gradient | Momentum |
| 2nd moment (variance) | $v_t$ | Magnitude/variance of gradient | Adaptive LR |

### Practical Decision Guide
```
Do you have sparse features (NLP bag-of-words)?
  → AdaGrad

Training an RNN?
  → RMSProp

Training a Transformer (BERT, GPT, ViT)?
  → AdamW (lr=1e-4, weight_decay=0.01)

Training a CNN (ResNet, EfficientNet)?
  → SGD + Momentum + Cosine LR (often beats Adam on final accuracy)
  → Or Adam for faster experimentation

Quick experiment / prototyping?
  → Adam with default settings (lr=1e-3)
```


---

## 8. Learning Rate Scheduling — The Most Impactful Hyperparameter

### Intuition: Drive Fast, Then Park Carefully
Imagine driving to a parking spot:
- **Far from the spot**: drive fast (large LR) — cover ground quickly
- **Near the spot**: slow down (small LR) — park precisely without overshooting

A fixed learning rate is like driving at the same speed the whole time:
- Too large: you keep driving past the parking spot (oscillates around minimum)
- Too small: takes forever to get there

**Learning rate scheduling** = change the learning rate during training automatically.

---

### 8.1 Step Decay

Reduce LR by a factor every N epochs:

$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / T_{\text{step}} \rfloor}$$

- $\gamma$ = decay factor (e.g., 0.1 or 0.5)
- $T_{\text{step}}$ = step size in epochs

Simple and widely used. Downside: abrupt drops, not smooth.

PyTorch: `StepLR(optimizer, step_size=30, gamma=0.1)`

---

### 8.2 Cosine Annealing

Smoothly decay LR following a cosine curve:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{\pi t}{T}\right)$$

- Starts high, smoothly decays to $\eta_{\min}$
- No abrupt drops — gradual slowdown
- **Very widely used** in modern training (ResNets, ViTs)

PyTorch: `CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)`

---

### 8.3 Cosine Annealing with Warm Restarts (SGDR)

Every $T_0$ epochs, restart the LR back to $\eta_{\max}$, with period doubling:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})\left(1 + \cos\frac{\pi T_{\text{cur}}}{T_i}\right)$$

- **Restarts** allow exploring new minima after each cycle
- Can escape sharp minima → often finds flatter (better-generalising) minima
- Period doubling: each cycle is longer than the previous

PyTorch: `CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)`

---

### 8.4 Linear Warmup (Critical for Transformers)

At the start of training, weight matrices are random. A large LR causes catastrophic early updates.

**Warmup**: start with a very small LR, linearly increase to target LR over $T_{\text{warm}}$ steps:

$$\eta_t = \eta_{\max} \cdot \frac{t}{T_{\text{warm}}} \quad \text{for } t \leq T_{\text{warm}}$$

After warmup, apply cosine decay (or another schedule).

**Why transformers need it:** attention weights initialised randomly; a large LR on step 1 creates degenerate attention distributions before any meaningful patterns form. Warmup gives the model a chance to stabilise.

---

### 8.5 LR Range Test (Smith, 2017)

Finding the right LR is hard. The **LR Range Test** automates it:
1. Start with a very small LR
2. Linearly increase LR over ~100 steps
3. Plot loss vs LR
4. Choose the LR just before the loss starts to explode

```python
# PyTorch Lightning has this built in:
# trainer.tune(model)

# Manual version:
lrs = np.logspace(-5, 0, 100)
for lr in lrs:
    # take one step with this lr, record loss
```

**Rule of thumb:** pick the LR at maximum gradient of the loss curve (steepest descent).

#### Interview Questions: LR Scheduling
> **Q: Why does LR scheduling improve results vs fixed LR?**
> A: High LR early = fast progress in loss landscape. Low LR late = fine-grained convergence to local minimum without oscillating. Neither extreme works alone; scheduling gets the best of both.

> **Q: Why is warmup important for transformers specifically?**
> A: Attention and feedforward weights are randomly initialised. A large initial LR creates degenerate attention patterns before any structure forms. Warmup allows the optimizer to take small, safe steps while the model develops initial structure.

> **Q: What is cosine annealing with warm restarts and why might it outperform standard cosine?**
> A: SGDR restarts LR periodically, allowing the optimizer to escape local minima it converged into, and explore new regions of the landscape in each cycle. This snapshot ensemble effect can find flatter minima that generalise better.

> **Q: MultiStepLR vs CosineAnnealingLR — when would you choose each?**
> A: StepLR/MultiStepLR: simple, interpretable, works when you know the epoch milestones from prior experiments. CosineAnnealingLR: better default, smooth decay, no need to choose milestones, works well for most modern architectures.


In [ ]:
# Figure 7: All LR Schedules — comparison
import numpy as np
import matplotlib.pyplot as plt

T = 200   # total epochs/steps
t = np.arange(T)

# ── Step Decay ──
eta_step = 0.1 * (0.1 ** (t // 50))

# ── Cosine Annealing ──
eta_min, eta_max = 1e-5, 0.1
eta_cosine = eta_min + 0.5*(eta_max - eta_min)*(1 + np.cos(np.pi * t / T))

# ── Cosine with Warm Restarts (T0=40, T_mult=2) ──
T0 = 40
def cosine_restart(t, T0, T_mult=2, eta_min=1e-5, eta_max=0.1):
    lrs = []
    cur_T = T0
    t_cur = 0
    for ti in t:
        if t_cur >= cur_T:
            t_cur = 0
            cur_T *= T_mult
        lr = eta_min + 0.5*(eta_max-eta_min)*(1+np.cos(np.pi*t_cur/cur_T))
        lrs.append(lr)
        t_cur += 1
    return np.array(lrs)
eta_restart = cosine_restart(t, T0=40)

# ── Warmup + Cosine ──
T_warm = 20
eta_warm_cosine = np.where(
    t < T_warm,
    eta_max * t / T_warm,
    eta_min + 0.5*(eta_max-eta_min)*(1 + np.cos(np.pi*(t-T_warm)/(T-T_warm)))
)

# ── Constant (baseline) ──
eta_const = np.full(T, 0.05)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

configs = [
    (eta_const,       'Constant LR (baseline)',            '#937860'),
    (eta_step,        'Step Decay (γ=0.1, every 50 ep)',   '#4C72B0'),
    (eta_cosine,      'Cosine Annealing',                  '#55A868'),
    (eta_restart,     'Cosine + Warm Restarts (SGDR)',      '#DD8452'),
    (eta_warm_cosine, 'Warmup (20 ep) + Cosine Decay',     '#8172B3'),
]
for i, (eta, title, col) in enumerate(configs):
    axes[i].plot(t, eta, color=col, lw=2.5)
    axes[i].set_xlabel('Epoch/Step')
    axes[i].set_ylabel('Learning Rate')
    axes[i].set_title(title)
    axes[i].set_ylim(-0.002, 0.115)
    if 'Warmup' in title:
        axes[i].axvspan(0, 20, alpha=0.12, color='gold', label='Warmup phase')
        axes[i].legend(fontsize=9)

# ── Combined plot ──
for eta, title, col in configs[1:]:
    axes[5].plot(t, eta, color=col, lw=2, label=title.split('(')[0].strip())
axes[5].legend(fontsize=8); axes[5].set_title('All Schedules: Comparison')
axes[5].set_xlabel('Epoch'); axes[5].set_ylabel('Learning Rate')

plt.suptitle('Figure 7: Learning Rate Schedules', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 9. PyTorch: Optimizer + Scheduler Demo

Putting it all together: training a real network with AdamW + cosine warmup.
We'll track loss, weight norms, learning rate, and gradient norms simultaneously.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

# ── Dataset: circles (non-linear classification) ──────────────────
N = 300
theta = torch.linspace(0, 4*np.pi, N)
r_inner = 0.5 + 0.1*torch.randn(N//2)
r_outer = 1.0 + 0.1*torch.randn(N//2)
X_inner = torch.stack([r_inner*torch.cos(theta[:N//2]),
                        r_inner*torch.sin(theta[:N//2])], dim=1)
X_outer = torch.stack([r_outer*torch.cos(theta[N//2:]),
                        r_outer*torch.sin(theta[N//2:])], dim=1)
X = torch.cat([X_inner, X_outer], dim=0)
y = torch.cat([torch.zeros(N//2), torch.ones(N//2)]).long()

# ── Model ─────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x): return self.net(x)

# ── Train with different optimizer+scheduler combos ───────────────
def train(optimizer_fn, scheduler_fn=None, epochs=150):
    model = MLP()
    opt = optimizer_fn(model.parameters())
    sch = scheduler_fn(opt) if scheduler_fn else None
    criterion = nn.CrossEntropyLoss()

    losses, accs, lrs, grad_norms = [], [], [], []
    for epoch in range(epochs):
        opt.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        # track gradient norm
        total_norm = sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None)**0.5
        grad_norms.append(total_norm.item())
        opt.step()
        if sch: sch.step()

        losses.append(loss.item())
        preds = out.argmax(dim=1)
        accs.append((preds == y).float().mean().item())
        lrs.append(opt.param_groups[0]['lr'])

    return model, losses, accs, lrs, grad_norms

T_warm = 20
epochs = 150

configs = [
    ("SGD",              lambda p: optim.SGD(p, lr=0.05, momentum=0.9),          None),
    ("Adam",             lambda p: optim.Adam(p, lr=1e-3),                        None),
    ("AdamW + Cosine",   lambda p: optim.AdamW(p, lr=1e-3, weight_decay=0.01),
     lambda o: optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs, eta_min=1e-6)),
    ("AdamW + Warmup+Cosine",
     lambda p: optim.AdamW(p, lr=1e-3, weight_decay=0.01),
     lambda o: optim.lr_scheduler.SequentialLR(
         o,
         [optim.lr_scheduler.LinearLR(o, start_factor=0.01, end_factor=1.0, total_iters=T_warm),
          optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs-T_warm, eta_min=1e-6)],
         milestones=[T_warm])),
]

results = {}
for name, opt_fn, sch_fn in configs:
    _, losses, accs, lrs, gnorms = train(opt_fn, sch_fn, epochs)
    results[name] = {'losses': losses, 'accs': accs, 'lrs': lrs, 'gnorms': gnorms}
    print(f"{name:30s} | Final loss: {losses[-1]:.4f} | Accuracy: {accs[-1]*100:.1f}%")

# ── Plots ──────────────────────────────────────────────────────────
colors = ['#C44E52','#4C72B0','#55A868','#8172B3']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for (name, r), col in zip(results.items(), colors):
    axes[0,0].plot(r['losses'], label=name, color=col, lw=2)
    axes[0,1].plot(r['accs'],   label=name, color=col, lw=2)
    axes[1,0].plot(r['lrs'],    label=name, color=col, lw=2)
    axes[1,1].plot(r['gnorms'], label=name, color=col, lw=2, alpha=0.75)

for ax, title, ylabel in [
    (axes[0,0], 'Training Loss',          'Cross-Entropy Loss'),
    (axes[0,1], 'Training Accuracy',      'Accuracy'),
    (axes[1,0], 'Learning Rate Schedule', 'Learning Rate'),
    (axes[1,1], 'Gradient Norm',          '||∇L||₂'),
]:
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

# ── Decision boundary ──────────────────────────────────────────────
# Re-train the best model
best_model, *_ = train(
    lambda p: optim.AdamW(p, lr=1e-3, weight_decay=0.01),
    lambda o: optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs, eta_min=1e-6),
    epochs=epochs
)
# (shown separately to avoid overloading the figure)

plt.suptitle('Figure 8: PyTorch Optimizer + Scheduler Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 10. Master Interview Q&A Cheatsheet

### Level 1 — Beginner

> **Q: Why can't we just use a very large learning rate for fast training?**
> A: A too-large LR causes parameter updates to overshoot the minimum — the loss oscillates or diverges. You need the LR to be small enough that each step reduces the loss. LR scheduling reduces it further as training progresses for fine-tuning.

> **Q: What does momentum do in SGD?**
> A: Maintains a velocity vector that accumulates in consistent gradient directions and dampens oscillations in noisy directions. Helps escape ravines in the loss landscape and converges faster.

> **Q: What is the difference between SGD and mini-batch SGD?**
> A: True SGD updates weights after one sample (very noisy). Mini-batch SGD processes a batch of B samples (typical: 32–256), averages their gradients, and then updates — far less noisy, exploits GPU parallelism.

> **Q: What is learning rate scheduling and why do we use it?**
> A: Systematically reducing (and sometimes cycling) the learning rate during training. High LR early = fast broad exploration; low LR late = precise convergence. Fixed LR is usually suboptimal.

---

### Level 2 — Mid-Level

> **Q: Explain Adam's two moments and how they interact in the update rule.**
> A: First moment ($m_t$, EMA of gradients) provides direction and momentum. Second moment ($v_t$, EMA of $g^2$) captures per-parameter gradient magnitude. The update $\frac{m_t}{\sqrt{v_t}+\epsilon}$ normalises momentum by gradient scale — effectively giving larger updates in directions with consistently small gradients, smaller in volatile directions.

> **Q: Why does Adam sometimes generalise worse than SGD?**
> A: Adam adapts to the training data and can converge to sharp minima that have great training loss but poor test loss. SGD+Momentum's higher noise has an implicit regularisation effect — it tends to find flatter minima. Several papers (Keskar et al., 2017; Wilson et al., 2017) demonstrated this. In practice: use Adam for speed, SGD+cosine for best final accuracy in vision.

> **Q: What is AdamW and why should it be preferred over Adam with weight_decay?**
> A: In Adam, adding weight decay via L2 adds $\lambda W$ to the gradient, which then gets scaled by the adaptive denominator. AdamW decouples weight decay — it's applied directly to weights at rate $\eta\lambda$, independent of gradient history. This ensures proper, consistent regularisation across all parameters.

> **Q: A model trained with Adam suddenly diverges after 10k steps. What do you check?**
> A: (1) Check LR — did the scheduler jump suddenly? (2) Check gradient norms per layer — is something exploding? (3) Check loss for NaN — divide by zero in activation function? (4) Check batch for degenerate examples. (5) Try adding gradient clipping (`max_norm=1.0`). (6) Try AdamW + weight decay for stability.

---

### Level 3 — Senior MLE / Staff Engineer

> **Q: Design a learning rate schedule for training a 7B parameter transformer from scratch.**
> A: (1) Linear warmup for ~2000 steps from lr=0 to peak lr ~3e-4 (or sqrt of batch size scaled). (2) Cosine decay to lr_min=3e-5 over remaining ~400k steps. (3) Use AdamW with β₁=0.9, β₂=0.95 (lower β₂ for large batch), ε=1e-8, weight_decay=0.1. (4) Clip gradients at max_norm=1.0. This is approximately the GPT-3 / LLaMA training recipe.

> **Q: What is the linear scaling rule for learning rate and when does it break?**
> A: If you multiply batch size by k, multiply learning rate by k (Goyal et al., 2017). Rationale: larger batch = lower gradient variance, so you can take larger steps. It breaks for very large batch sizes (>8k) — the linear approximation breaks down because curvature becomes significant. Solution: use LARS/LAMB optimizer for ultra-large batches.

> **Q: What is SAM (Sharpness-Aware Minimisation) and how does it relate to LR scheduling?**
> A: SAM finds parameters $W$ that lie in a flat loss neighbourhood by perturbing weights in the gradient direction, computing the loss at the perturbed point, then updating toward a solution that simultaneously minimises loss and its maximum perturbation. It explicitly seeks flat minima. LR scheduling reaches flat minima indirectly (by reducing LR ensures settling in a basin floor). SAM is more principled but 2× the compute cost.

> **Q: Compare Adam, LAMB, and LARS — when would you use each?**
> A: Adam: standard adaptive optimizer, works up to ~8k batch size. LARS (Layer-wise Adaptive Rate Scaling): used for very large batch SGD (ImageNet in 76 minutes). LAMB: Adam equivalent of LARS, per-layer adaptive rate scaling — used for BERT pretraining with batch size 64k in 76 minutes. The key insight in LARS/LAMB: normalise gradient by its L2 norm per layer, preventing large layers from dominating small ones at mega-batch scales.
